# STIL 2023 Scraper para Colab

Este notebook extrai os artigos do STIL 2023 a partir do `dblp`, segue o link `view` para a página da SBC, baixa os PDFs e gera um JSON no formato esperado.

In [1]:
!pip install -q requests beautifulsoup4 pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.5 MB/s eta 0:00:00


## Etapa 1: dependências

Esta célula instala as bibliotecas usadas no notebook:
- `requests`: faz as requisições HTTP
- `beautifulsoup4`: interpreta o HTML
- `pypdf`: lê os PDFs e extrai o texto

In [2]:
import json
import re
import unicodedata
from dataclasses import dataclass
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Dict, List, Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader


DBLP_TOC_URL = "https://dblp.org/db/conf/stil/stil2023.html"
REQUEST_TIMEOUT = 30
USER_AGENT = "Mozilla/5.0 (compatible; STIL2023Scraper/1.0)"
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

STOPWORDS = {
    "en": {
        "the": "DET", "a": "DET", "an": "DET", "this": "DET", "that": "DET",
        "these": "DET", "those": "DET", "of": "ADP", "in": "ADP", "on": "ADP",
        "for": "ADP", "to": "PART", "and": "CCONJ", "or": "CCONJ", "but": "CCONJ",
        "with": "ADP", "without": "ADP", "by": "ADP", "from": "ADP", "is": "AUX",
        "are": "AUX", "was": "AUX", "were": "AUX", "be": "AUX", "been": "AUX",
        "being": "AUX", "it": "PRON", "its": "DET", "we": "PRON", "our": "DET",
        "they": "PRON", "their": "DET"
    },
    "pt": {
        "o": "DET", "a": "DET", "os": "DET", "as": "DET", "um": "DET", "uma": "DET",
        "uns": "DET", "umas": "DET", "de": "ADP", "do": "ADP", "da": "ADP", "dos": "ADP",
        "das": "ADP", "em": "ADP", "no": "ADP", "na": "ADP", "nos": "ADP", "nas": "ADP",
        "para": "ADP", "por": "ADP", "com": "ADP", "sem": "ADP", "e": "CCONJ", "ou": "CCONJ",
        "mas": "CCONJ", "é": "AUX", "são": "AUX", "foi": "AUX", "ser": "AUX", "se": "PRON",
        "que": "SCONJ", "eu": "PRON", "nós": "PRON", "eles": "PRON", "elas": "PRON"
    },
}


@dataclass
class DblpEntry:
    """Estrutura tempor?ria com os dados b?sicos extra?dos da p?gina ?ndice do dblp."""
    title: str
    ee_url: Optional[str]
    details_url: Optional[str]
    authors: List[Dict[str, Optional[str]]]


def fix_mojibake(value: str) -> str:
    """Tenta corrigir texto com caracteres corrompidos por problemas de encoding."""
    if not value:
        return value
    suspicious = ("Ã", "Â", "â", "Ë", "Ê", "´", "€", "™")
    if not any(char in value for char in suspicious):
        return value
    try:
        repaired = value.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value
    original_hits = sum(value.count(char) for char in suspicious)
    repaired_hits = sum(repaired.count(char) for char in suspicious)
    return repaired if repaired_hits < original_hits else value


def normalize_whitespace(value: str) -> str:
    """Remove espa?os duplicados e aplica corre??o de encoding quando necess?rio."""
    return re.sub(r"\s+", " ", fix_mojibake(value or "")).strip()


def normalize_key(value: str) -> str:
    """Normaliza texto para compara??es est?veis, removendo acentos e padronizando caixa."""
    normalized = unicodedata.normalize("NFKD", value or "")
    ascii_only = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    return normalize_whitespace(ascii_only).casefold()


def infer_language_from_code(code: Optional[str], fallback_title: str) -> str:
    """Converte o c?digo de idioma para o r?tulo final usado no JSON."""
    if code == "pt":
        return "Português"
    if code == "en":
        return "Inglês"
    normalized = normalize_key(fallback_title)
    pt_markers = (" de ", " do ", " da ", " em ", " para ", " portugues", " analise ")
    return "Português" if any(marker in f" {normalized} " for marker in pt_markers) else "Inglês"


def format_date(date_value: Optional[str]) -> str:
    """Converte datas para o formato dd/mm/aaaa quando poss?vel."""
    if not date_value:
        return ""
    for fmt in ("%Y/%m/%d", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(date_value, fmt).strftime("%d/%m/%Y")
        except ValueError:
            pass
    return date_value


def split_references(ref_block: Optional[BeautifulSoup]) -> List[str]:
    """Separa o bloco HTML de refer?ncias em uma lista de refer?ncias textuais."""
    if ref_block is None:
        return []
    html_content = ref_block.decode_contents()
    parts = re.split(r"<br\s*/?>\s*<br\s*/?>", html_content, flags=re.IGNORECASE)
    references = []
    for part in parts:
        text = normalize_whitespace(BeautifulSoup(part, "html.parser").get_text(" ", strip=True))
        if text:
            references.append(text)
    return references


def extract_pdf_text(content: bytes) -> str:
    """Extrai e concatena o texto de todas as p?ginas de um PDF."""
    reader = PdfReader(BytesIO(content))
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        if text:
            pages.append(text)
    return normalize_whitespace("\n".join(pages))


def heuristic_pos(token: str, language_code: str) -> str:
    """Atribui uma classe gramatical simples ao token usando regras heur?sticas."""
    lowered = token.casefold()
    if re.fullmatch(r"\d+(?:[.,]\d+)*", token):
        return "NUM"
    if re.fullmatch(r"[^\w\s]", token):
        return "PUNCT"
    if lowered in STOPWORDS.get(language_code, {}):
        return STOPWORDS[language_code][lowered]
    if token[:1].isupper():
        return "PROPN"
    if lowered.endswith(("mente", "ly")):
        return "ADV"
    if lowered.endswith(("ar", "er", "ir", "ing", "ed")):
        return "VERB"
    if lowered.endswith(("al", "vel", "ivo", "iva", "ous", "ful", "able", "ible")):
        return "ADJ"
    return "NOUN"


def heuristic_lemma(token: str) -> str:
    """Gera um lema simplificado a partir da forma normalizada do token."""
    if re.fullmatch(r"[^\w\s]", token):
        return token
    return normalize_key(token)


def tokenize_with_annotations(text: str, language_label: str) -> Dict[str, List[str]]:
    """Tokeniza o texto e produz tokens, POS tags heur?sticas e lemas."""
    language_code = "pt" if language_label == "Português" else "en"
    tokens = TOKEN_PATTERN.findall(text)
    pos_tags = [heuristic_pos(token, language_code) for token in tokens]
    lemmas = [heuristic_lemma(token) for token in tokens]
    return {
        "artigo_tokenizado": tokens,
        "pos_tagger": pos_tags,
        "lema": lemmas,
    }


def parse_dblp_toc(session: requests.Session) -> List[DblpEntry]:
    """L? a p?gina do dblp e retorna a lista de artigos do evento com metadados b?sicos."""
    response = session.get(DBLP_TOC_URL, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    entries = []
    for item in soup.select("li.entry.inproceedings"):
        title_node = item.select_one("cite span.title")
        if title_node is None:
            continue
        ee_link = item.select_one("li.ee a[itemprop='url']")
        details_link = item.select_one("li.details a")
        authors = []
        for author_node in item.select("cite span[itemprop='author']"):
            name_node = author_node.select_one("span[itemprop='name']")
            if name_node is None:
                continue
            orcid_node = author_node.select_one("img[title]")
            authors.append({
                "nome": normalize_whitespace(name_node.get_text(" ", strip=True)),
                "afiliacao": "",
                "orcid": f"http://orcid.org/{orcid_node.get('title')}" if orcid_node and orcid_node.get("title") else "",
            })
        entries.append(DblpEntry(
            title=normalize_whitespace(title_node.get_text(" ", strip=True)).rstrip("."),
            ee_url=ee_link.get("href") if ee_link else None,
            details_url=urljoin("https://dblp.org/", details_link.get("href")) if details_link else None,
            authors=authors,
        ))
    return entries


def parse_article_page(session: requests.Session, entry: DblpEntry, storage_key: str, download_dir: Path) -> Dict[str, object]:
    """Extrai os dados completos de um artigo na SBC, baixa o PDF e monta o JSON do item."""
    if not entry.ee_url:
        raise ValueError(f"Entrada sem link eletrônico: {entry.title}")
    response = session.get(entry.ee_url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    article_url = response.url

    meta = {}
    for node in soup.select("meta[name], meta[property]"):
        key = node.get("name") or node.get("property")
        content = node.get("content", "")
        if key:
            meta.setdefault(key, []).append(content)

    title = normalize_whitespace((meta.get("citation_title") or [entry.title])[0]).rstrip(".")
    language_code = (meta.get("DC.Language") or [""])[0].strip().lower()
    language_label = infer_language_from_code(language_code, title)

    authors = []
    for author_node in soup.select("ul.item.authors > li"):
        name_node = author_node.select_one("span.name")
        if name_node is None:
            continue
        affiliation_node = author_node.select_one("span.affiliation")
        orcid_node = author_node.select_one("span.orcid a")
        authors.append({
            "nome": normalize_whitespace(name_node.get_text(" ", strip=True)),
            "afiliacao": normalize_whitespace(affiliation_node.get_text(" ", strip=True)) if affiliation_node else "",
            "orcid": orcid_node.get("href", "") if orcid_node else "",
        })

    if not authors:
        authors = entry.authors

    dblp_orcid_by_name = {normalize_key(author["nome"]): author["orcid"] for author in entry.authors}
    for author in authors:
        if not author["orcid"]:
            author["orcid"] = dblp_orcid_by_name.get(normalize_key(author["nome"]), "")

    abstract_node = soup.select_one("div.item.abstract")
    abstract_text = ""
    if abstract_node is not None:
        label = abstract_node.select_one(".label")
        if label:
            label.extract()
        abstract_text = normalize_whitespace(abstract_node.get_text(" ", strip=True))
    if not abstract_text:
        abstract_text = normalize_whitespace((meta.get("DC.Description") or [""])[0])

    keywords_node = soup.select_one("div.item.keywords span.value")
    keywords = []
    if keywords_node is not None:
        keywords = [normalize_whitespace(keyword) for keyword in keywords_node.get_text(" ", strip=True).split(",")]
        keywords = [keyword for keyword in keywords if keyword]

    references_node = soup.select_one("div.item.references div.value")
    references = split_references(references_node)

    pdf_url = (meta.get("citation_pdf_url") or [""])[0]
    article_text = ""
    if pdf_url:
        pdf_response = session.get(pdf_url, timeout=REQUEST_TIMEOUT)
        pdf_response.raise_for_status()
        pdf_path = download_dir / Path(storage_key).name
        pdf_path.write_bytes(pdf_response.content)
        article_text = extract_pdf_text(pdf_response.content)

    annotations = tokenize_with_annotations(article_text, language_label)

    return {
        "titulo": title,
        "informacoes_url": article_url,
        "idioma": language_label,
        "storage_key": storage_key,
        "autores": authors,
        "data_publicacao": format_date((meta.get("citation_date") or [""])[0]),
        "resumo": abstract_text,
        "keywords": keywords,
        "referencias": references,
        "artigo_completo": article_text,
        "artigo_tokenizado": annotations["artigo_tokenizado"],
        "pos_tagger": annotations["pos_tagger"],
        "lema": annotations["lema"],
    }


def build_dataset(output_path: Path, download_dir: Path, limit: Optional[int] = None) -> List[Dict[str, object]]:
    """Coordena a coleta completa, salva os arquivos e retorna a lista final de artigos."""
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    download_dir.mkdir(parents=True, exist_ok=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    entries = parse_dblp_toc(session)
    if limit is not None:
        entries = entries[:limit]

    dataset = []
    for index, entry in enumerate(entries, start=1):
        storage_key = f"files/article_{index:03d}.pdf"
        article = parse_article_page(session, entry, storage_key, download_dir)
        dataset.append(article)
        print(f"[{index}/{len(entries)}] extraido: {article['titulo']}")

    output_path.write_text(json.dumps(dataset, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Arquivo gerado em: {output_path}")
    return dataset

## Etapa 2: definição do scraper

Esta célula contém toda a lógica do projeto.

Fluxo interno:
1. Lê a página do `dblp` para encontrar os artigos do STIL 2023.
2. Para cada artigo, segue o link `view` até a página da SBC.
3. Extrai título, autores, afiliação, ORCID, resumo, palavras-chave, referências e data.
4. Baixa o PDF e extrai o texto completo.
5. Gera tokenização, POS tagging heurístico e lema heurístico.
6. Salva tudo em JSON.

Observação: `pos_tagger` e `lema` usam regras simples, não um modelo linguístico avançado.

## O que cada função faz

- `fix_mojibake(value)`: tenta corrigir textos com caracteres quebrados por problema de encoding.
- `normalize_whitespace(value)`: remove espaços duplicados e aplica a correção de encoding.
- `normalize_key(value)`: normaliza um texto para comparação, removendo acentos e padronizando caixa.
- `infer_language_from_code(code, fallback_title)`: converte o código de idioma em `Português` ou `Inglês`.
- `format_date(date_value)`: transforma datas para o formato `dd/mm/aaaa`.
- `split_references(ref_block)`: separa o bloco HTML de referências em uma lista de strings.
- `extract_pdf_text(content)`: lê o PDF e junta o texto de todas as páginas em uma única string.
- `heuristic_pos(token, language_code)`: atribui uma classe gramatical simples ao token usando regras heurísticas.
- `heuristic_lemma(token)`: gera um lema simplificado para o token.
- `tokenize_with_annotations(text, language_label)`: tokeniza o texto e produz `artigo_tokenizado`, `pos_tagger` e `lema`.
- `parse_dblp_toc(session)`: acessa a página do `dblp` e extrai a lista de artigos do evento.
- `parse_article_page(session, entry, storage_key, download_dir)`: acessa a página da SBC de um artigo, extrai os metadados, baixa o PDF e monta o JSON daquele artigo.
- `build_dataset(output_path, download_dir, limit)`: coordena o processo completo, percorrendo todos os artigos e salvando o arquivo final.

Classe auxiliar:
- `DblpEntry`: estrutura temporária usada para guardar os dados básicos vindos do `dblp` antes do enriquecimento na página da SBC.

In [3]:
# Use limit=1 para teste rápido. Use limit=None para processar todos.
dataset = build_dataset(
    output_path=Path("output/stil2023_articles.json"),
    download_dir=Path("files"),
    limit=30,
)

print(f"Total de artigos extraídos: {len(dataset)}")

[1/30] extraido: Less is More? Investigating Meta-Learning’s Suitability in Sentence Compression for Low-Resource Data
[2/30] extraido: Predição de transtorno depressivo em redes sociais: BERT supervisionado ou ChatGPT zero-shot?
[3/30] extraido: Semantic Textual Similarity: In Defense of Wordnet-Based Methods
[4/30] extraido: Contextual stance classification using prompt engineering
[5/30] extraido: Classificação de gêneros a partir de letras de músicas em português
[6/30] extraido: Sexismo no Brasil: análise de um Word Embedding por meio de testes baseados em associação implícita
[7/30] extraido: Etiquetagem morfossintática multigênero para o português do Brasil segundo o modelo "Universal Dependencies"
[8/30] extraido: Automated question answering via natural language sentence similarity: Achievements for Brazilian e-commerce platforms
[9/30] extraido: Classificação de Polaridade Orientada aos Alvos de Opinião em Comentários sobre Debate Político em Português
[10/30] extraido: How G

## Etapa 3: executar a coleta

Nesta célula você escolhe quantos artigos deseja processar.

- `limit=1`: teste rápido
- `limit=30`: coleta 30 artigos
- `limit=None`: coleta todos os artigos disponíveis na página

As pastas `files/` e `output/` são criadas automaticamente, se ainda não existirem.

In [4]:
dataset[0]

{'titulo': 'Less is More? Investigating Meta-Learning’s Suitability in Sentence Compression for Low-Resource Data',
 'informacoes_url': 'https://sol.sbc.org.br/index.php/stil/article/view/25432',
 'idioma': 'Inglês',
 'storage_key': 'files/article_001.pdf',
 'autores': [{'nome': 'L. Gustavo Coutinho do R.',
   'afiliacao': 'UFC',
   'orcid': 'http://orcid.org/0000-0003-2039-3297'},
  {'nome': 'José Antônio F. de Macêdo', 'afiliacao': 'UFC', 'orcid': ''},
  {'nome': 'Ticiana L. Coelho da Silva', 'afiliacao': 'UFC', 'orcid': ''}],
 'data_publicacao': '25/09/2023',
 'resumo': 'The sentence compression task is essential in the text summarization process. Unfortunately, the lack of labeled data for specific domains restricts the training of deep learning models to address this problem effectively. In this paper, we present an approach using a meta-learning algorithm called MAML to tackle this issue and assess the viability of this technique for the given task, with particular emphasis on it

## Etapa 4: inspecionar o resultado

Esta célula mostra o primeiro item do dataset gerado para você validar o formato do JSON.

In [5]:
from google.colab import files

files.download("output/stil2023_articles.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Etapa 5: baixar o arquivo final

Depois que a coleta terminar, esta célula baixa o arquivo `output/stil2023_articles.json` para sua máquina.

Se quiser, você também pode navegar pelos arquivos do Colab e baixar manualmente os PDFs salvos em `files/`.